In [1]:
import pandas as pd
import joblib
import json

def generate_full_persona_reports():
    print("[*] Loading ML models and data...")
    try:
        df = pd.read_csv("cf_ml_features_clean.csv")
        scaler = joblib.load('persona_scaler.pkl')
        gmm = joblib.load('persona_gmm_model.pkl')
    except Exception as e:
        print(f"[-] Error loading files: {e}")
        return

    features_to_drop = ['handle', 'total_submissions']
    if 'dbscan_label' in df.columns:
        features_to_drop.append('dbscan_label')
        
    X = df.drop(columns=features_to_drop)
    feature_names = X.columns
    
    # Predict clusters
    X_scaled = scaler.transform(X)
    df['Persona_Cluster'] = gmm.predict(X_scaled)
    
    tag_columns = {
        'math_pref': 'Math', 
        'dp_pref': 'Dynamic Programming', 
        'graph_pref': 'Graphs/Trees', 
        'brute_pref': 'Brute Force/Impl',
        'greedy_pref': 'Greedy/Two Pointers',
        'binary_pref': 'Binary Search',
        'cons_pref': 'Constructive',
        'datastruct_pref': 'Data Structures'
    }
    
    global_means = df[list(tag_columns.keys())].mean()
    cluster_means = df.groupby('Persona_Cluster')[feature_names].mean()
    
    master_profiles = {}

    print("\n" + "="*60)
    print("THE DEEP-DIVE PERSONA REPORT")
    print("="*60)
    
    for cluster_id in range(10):
        users_in_cluster = len(df[df['Persona_Cluster'] == cluster_id])
        profile = cluster_means.loc[cluster_id]
        
        # Calculate all lifts
        lifts = []
        for col, human_name in tag_columns.items():
            lift_val = profile[col] / global_means[col] if global_means[col] > 0 else 0
            lifts.append({"domain": human_name, "lift": round(lift_val, 2)})
            
        # Sort domains from best to worst
        lifts = sorted(lifts, key=lambda x: x['lift'], reverse=True)
        
        print(f"\n[ Archetype {cluster_id} ] - ({users_in_cluster} users)")
        print(f"  [ CORE METRICS ]")
        print(f"  -> Accuracy:           {profile['accuracy']*100:.1f}%")
        print(f"  -> One-Shot Precision: {profile['one_shot_rate']*100:.1f}%")
        print(f"  -> TLE Rate:           {profile['optimization_struggle']*100:.1f}%")
        print(f"  -> Abandonment Rate:   {profile['abandonment_rate']*100:.1f}%")
        print(f"  -> Tilt Speed:         {profile['tilt_speed_seconds']:.0f} seconds")
        print(f"  -> Avg Solved Rating:  {profile['avg_solved_rating']:.0f}")
        
        print(f"  [ DOMAIN RADAR (Lift) ]")
        for i in range(3): # Print Top 3
            print(f"  -> Strength: {lifts[i]['domain']} ({lifts[i]['lift']}x)")
        print(f"  -> Weakness: {lifts[-1]['domain']} ({lifts[-1]['lift']}x)")
        
        # Save to dictionary for JSON export
        master_profiles[int(cluster_id)] = {
            "size": users_in_cluster,
            "accuracy": round(profile['accuracy'], 3),
            "one_shot": round(profile['one_shot_rate'], 3),
            "tle_rate": round(profile['optimization_struggle'], 3),
            "abandonment": round(profile['abandonment_rate'], 3),
            "tilt_speed": round(profile['tilt_speed_seconds'], 1),
            "avg_rating": round(profile['avg_solved_rating'], 0),
            "strengths": [lifts[0]['domain'], lifts[1]['domain']],
            "weakness": lifts[-1]['domain']
        }

    # Save to JSON
    with open("persona_profiles.json", "w") as f:
        json.dump(master_profiles, f, indent=4)
        
    print("\n[+] Full statistical profiles saved to 'persona_profiles.json'")

if __name__ == "__main__":
    generate_full_persona_reports()

[*] Loading ML models and data...

THE DEEP-DIVE PERSONA REPORT

[ Archetype 0 ] - (85 users)
  [ CORE METRICS ]
  -> Accuracy:           61.5%
  -> One-Shot Precision: 66.8%
  -> TLE Rate:           2.6%
  -> Abandonment Rate:   29.9%
  -> Tilt Speed:         256 seconds
  -> Avg Solved Rating:  1098
  [ DOMAIN RADAR (Lift) ]
  -> Strength: Brute Force/Impl (1.13x)
  -> Strength: Math (1.04x)
  -> Strength: Constructive (1.02x)
  -> Weakness: Data Structures (0.64x)

[ Archetype 1 ] - (179 users)
  [ CORE METRICS ]
  -> Accuracy:           45.9%
  -> One-Shot Precision: 53.6%
  -> TLE Rate:           7.5%
  -> Abandonment Rate:   34.9%
  -> Tilt Speed:         255 seconds
  -> Avg Solved Rating:  1176
  [ DOMAIN RADAR (Lift) ]
  -> Strength: Greedy/Two Pointers (1.08x)
  -> Strength: Math (1.05x)
  -> Strength: Brute Force/Impl (1.04x)
  -> Weakness: Data Structures (0.93x)

[ Archetype 2 ] - (103 users)
  [ CORE METRICS ]
  -> Accuracy:           51.8%
  -> One-Shot Precision: 57.1%
